# Attention-Head Trajectories Production Run

This notebook runs a **production-scale experiment** of the head-trajectories pipeline.

It is designed to run from the repository on any machine with sufficient GPU resources.

- 15M parameter model (8 layers, 8 heads)
- real text dataset (WikiText-103)
- dense checkpoint schedule (60 checkpoints)
- full probing and trajectory analysis
- publication-quality outputs

The goal is to produce results suitable for the full-scale study before launching the complete OpenWebText runs.


## Before You Run

Runtime assumptions:

1. You can run this notebook either inside the repository or from an arbitrary working directory.
2. If the repository is not already present, the setup cell will clone it automatically.
3. If you have GPU access, the notebook will use it automatically.
4. The default config targets a production-scale run with batch_size=192.
5. If you are on a smaller GPU, reduce `batch_size`, `probe_batch_size`, or `total_steps` in the config cell.

Expected behavior:

- training: production-scale 15M parameter model
- probing: full probe sets (500 general, 100 induction, 50 positional)
- artifacts: saved locally under an absolute path printed by the notebook

Dataset choice:

- We use **`Salesforce/wikitext` with `wikitext-103-raw-v1`** by default.
- It is a **real dataset**, not synthetic.
- It keeps case, punctuation, and numbers, and is commonly used for language modeling.
- It is still much smaller than OpenWebText, but large enough for meaningful results.

References:

- Hugging Face dataset page: https://huggingface.co/datasets/Salesforce/wikitext
- Salesforce WikiText overview: https://www.salesforce.com/blog/the-wikitext-long-term-dependency-language-modeling-dataset/


In [ ]:
# Local / Modal-friendly setup
# This cell either finds the repository root from the current working directory
# or clones the repo locally if it is not already present.

import sys
import subprocess
import importlib.util
from pathlib import Path

REPO_URL = 'https://github.com/abderahmane-ai/head-trajectories.git'
REPO_DIRNAME = 'head-trajectories'

start_dir = Path.cwd().resolve()
candidates = [start_dir] + list(start_dir.parents)
ROOT = next(
    (p for p in candidates if (p / 'model').exists() and (p / 'probing').exists() and (p / 'training').exists()),
    None,
)

if ROOT is None:
    clone_root = start_dir / REPO_DIRNAME
    if not clone_root.exists():
        !git clone {REPO_URL} {clone_root}
    ROOT = clone_root.resolve()

if not ((ROOT / 'model').exists() and (ROOT / 'probing').exists() and (ROOT / 'training').exists()):
    raise RuntimeError(f'Repository root is invalid: {ROOT}')

%cd {ROOT}

required_modules = {
    'datasets': 'datasets',
    'tiktoken': 'tiktoken',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
}
missing_packages = [pkg for pkg, module_name in required_modules.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Notebook runtime dependencies already available.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Repository root:', ROOT)


## Production Run Configuration

This notebook is configured for a **full-scale 15M parameter production run**:

**Model architecture (15M params):**
- 8 layers × 8 heads
- d_model = 384, d_ffn = 1536
- block_size = 256

**Training configuration:**
- 60,000 optimization steps
- batch_size = 192
- ~60 checkpoints with dense early sampling
- early stopping after 5 non-improving checkpoints (min 10K steps)

**Probe configuration:**
- 350 general sequences
- 70 induction probes
- 35 positional pairs
- probe_batch_size = 64

This configuration produces publication-quality results suitable for the full OpenWebText experiment.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import math
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset

from model import ModelConfig, TransformerLM
from data.loader import get_tokenizer
from data.probe import (
    build_general_probes,
    build_induction_probes,
    build_positional_probes,
    verify_induction_probes,
)
from data.calibration import calibrate_thresholds
from training.scheduler import CosineScheduler
from training.trainer import save_checkpoint
from probing.extractor import extract_checkpoint
from probing.pipeline import discover_checkpoints, score_all_heads
from probing.classifier import HeadClassifier, HEAD_TYPES
from analysis.trajectories import (
    compute_global_curves,
    compute_specialization_onset,
    compute_head_trajectories,
    find_interesting_trajectories,
)


@dataclass
class PilotConfig:
    seed: int = 42
    dataset_name: str = 'Salesforce/wikitext'
    dataset_config: str = 'wikitext-103-raw-v1'
    block_size: int = 256
    n_layers: int = 8
    n_heads: int = 8
    d_model: int = 384
    d_ffn: int = 1536
    batch_size: int = 192
    total_steps: int = 60000
    warmup_steps: int = 1000
    max_lr: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    train_eval_batches: int = 20
    probe_batch_size: int = 64
    n_general: int = 350
    n_induction: int = 70
    n_pairs: int = 35
    n_calibration_seeds: int = 3
    checkpoint_steps: tuple = (0, 50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 800, 900, 1000, 1200, 1400, 1600, 1800, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000, 22000, 24000, 26000, 28000, 30000, 32000, 34000, 36000, 38000, 40000, 42000, 44000, 46000, 48000, 50000, 52000, 54000, 56000, 58000, 60000)
    early_stopping_patience_ckpts: int = 5
    min_steps_before_early_stop: int = 10000
    run_name: str = 'production_15m'
    artifact_root: Path = ROOT / 'pilot_artifacts' / 'production_15m'

    @property
    def ckpt_dir(self) -> Path:
        return self.artifact_root / 'checkpoints' / self.run_name

    @property
    def probe_path(self) -> Path:
        return self.artifact_root / 'probe' / 'probe_dataset_small.pt'

    @property
    def results_path(self) -> Path:
        return self.artifact_root / 'results' / f'results_{self.run_name}.pt'

    @property
    def best_ckpt_path(self) -> Path:
        return self.artifact_root / 'checkpoints' / f'{self.run_name}_best.pt'

    @property
    def ties_path(self) -> Path:
        return self.artifact_root / 'results' / f'ties_{self.run_name}.csv'

    @property
    def figure_path(self) -> Path:
        return self.artifact_root / 'figures' / 'pilot_global_curves.png'


CFG = PilotConfig()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENC = get_tokenizer()
RESET_ARTIFACTS = False

if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

if RESET_ARTIFACTS and CFG.artifact_root.exists():
    shutil.rmtree(CFG.artifact_root)
    print('Removed existing artifacts at:', CFG.artifact_root)

for path in [CFG.ckpt_dir, CFG.probe_path.parent, CFG.results_path.parent, CFG.figure_path.parent, CFG.best_ckpt_path.parent]:
    path.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print('GPU:', torch.cuda.get_device_name(0), f'({total_gb:.1f} GB)')
print('Artifacts:', CFG.artifact_root.resolve())
print('Checkpoint dir:', CFG.ckpt_dir.resolve())
print('Best checkpoint path:', CFG.best_ckpt_path.resolve())
print('Probe path:', CFG.probe_path.resolve())
print('Results path:', CFG.results_path.resolve())


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)

pilot_model_config = ModelConfig(
    vocab_size=50257,
    block_size=CFG.block_size,
    n_layers=CFG.n_layers,
    n_heads=CFG.n_heads,
    d_model=CFG.d_model,
    d_ffn=CFG.d_ffn,
)

pilot_model = TransformerLM(pilot_model_config)
print(pilot_model_config)
print(f'Actual trainable params: {pilot_model.count_parameters() / 1e6:.2f}M')


## Load a Real Dataset and Tokenize It

We use the three standard splits as follows:

- `train`: training tokens
- `validation`: validation-loss estimation during training
- `test`: held-out source for the probe sequences

That keeps the pilot honest: we do not build probes from the training split.


In [ ]:
dataset = load_dataset(CFG.dataset_name, CFG.dataset_config)
dataset


In [ ]:
def encode_split_texts(texts, encoder):
    flat_tokens = []
    for text in texts:
        if text and text.strip():
            flat_tokens.extend(encoder.encode_ordinary(text))
            flat_tokens.append(encoder.eot_token)
    return torch.tensor(flat_tokens, dtype=torch.long)


def tokens_to_fixed_sequences(tokens: torch.Tensor, block_size: int):
    usable = (len(tokens) // block_size) * block_size
    seqs = tokens[:usable].view(-1, block_size)
    return seqs


train_tokens = encode_split_texts(dataset['train']['text'], ENC)
val_tokens = encode_split_texts(dataset['validation']['text'], ENC)
test_tokens = encode_split_texts(dataset['test']['text'], ENC)

probe_source_sequences = tokens_to_fixed_sequences(test_tokens, CFG.block_size)

print(f'train tokens: {len(train_tokens):,}')
print(f'validation tokens: {len(val_tokens):,}')
print(f'test tokens: {len(test_tokens):,}')
print(f'probe source sequences: {probe_source_sequences.shape}')


## Build Production Probe Dataset

We keep the same probe types as the main project with production-scale sizes:

- 500 general sequences
- 100 induction probes
- 50 positional pairs

We also calibrate thresholds on a randomly initialized model, exactly like the main project.


In [ ]:
def build_small_probe_dict(raw_sequences, cfg: PilotConfig):
    raw_sequences = [seq.tolist() for seq in raw_sequences]
    rng = random.Random(cfg.seed)
    rng.shuffle(raw_sequences)

    general_pool_size = cfg.n_general * 2
    induction_pool_size = cfg.n_induction * 3
    positional_pool_size = cfg.n_pairs * 4
    total_needed = general_pool_size + induction_pool_size + positional_pool_size

    if len(raw_sequences) < total_needed:
        raise RuntimeError(f'Need at least {total_needed} source sequences, found {len(raw_sequences)}')

    offset = 0
    pool_general = raw_sequences[offset: offset + general_pool_size]
    offset += general_pool_size
    pool_induction = raw_sequences[offset: offset + induction_pool_size]
    offset += induction_pool_size
    pool_positional = raw_sequences[offset: offset + positional_pool_size]

    general_seqs = build_general_probes(pool_general, n_seqs=cfg.n_general, block_size=cfg.block_size, seed=cfg.seed)
    induction_seqs, induction_p1, induction_p2 = build_induction_probes(
        pool_induction,
        n_probes=cfg.n_induction,
        subseq_len_range=(5, 10),
        block_size=cfg.block_size,
        seed=cfg.seed,
    )
    positional_seqs, positional_pairs = build_positional_probes(
        pool_positional,
        n_pairs=cfg.n_pairs,
        block_size=cfg.block_size,
        seed=cfg.seed,
    )

    probe_dict = {
        'general_seqs': general_seqs,
        'induction_seqs': induction_seqs,
        'induction_p1': induction_p1,
        'induction_p2': induction_p2,
        'positional_seqs': positional_seqs,
        'positional_pairs': positional_pairs,
        'creation_seed': torch.tensor(cfg.seed, dtype=torch.long),
        'block_size': torch.tensor(cfg.block_size, dtype=torch.long),
    }
    return probe_dict


probe_dict = build_small_probe_dict(probe_source_sequences, CFG)
verify_induction_probes(probe_dict)

# Calibrate thresholds from random baseline (matches main codebase methodology)
raw_thresholds_mean, thresholds_std, thresholds_per_seed = calibrate_thresholds(
    probe_dict=probe_dict,
    config=pilot_model_config,
    device=DEVICE,
    batch_size=CFG.probe_batch_size,
    n_seeds=CFG.n_calibration_seeds,
)

# Use raw calibration thresholds directly (no manual clamping)
probe_dict['pilot_thresholds'] = torch.tensor(raw_thresholds_mean, dtype=torch.float32)
probe_dict['pilot_thresholds_std'] = torch.tensor(thresholds_std, dtype=torch.float32)
probe_dict['pilot_thresholds_per_seed'] = torch.tensor(thresholds_per_seed, dtype=torch.float32)

torch.save(probe_dict, CFG.probe_path)
print('Saved probe dataset to:', CFG.probe_path)
print('Calibrated thresholds (raw):', raw_thresholds_mean)
print('Thresholds std:', thresholds_std)


## Train Production Model

Training configuration:

- single seed (42)
- 60,000 optimization steps
- dense early checkpoints, then wider spacing (~60 total)
- mixed precision on GPU when available
- best-checkpoint tracking on validation loss
- early stopping when validation keeps worsening across checkpoints

This is a production-scale run suitable for publication-quality results.


In [ ]:
def sample_batch(tokens: torch.Tensor, batch_size: int, block_size: int, device: torch.device):
    max_start = len(tokens) - block_size - 1
    if max_start <= 0:
        raise ValueError('Token tensor is too short for the requested block size.')
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([tokens[i:i + block_size] for i in starts]).to(device)
    y = torch.stack([tokens[i + 1:i + block_size + 1] for i in starts]).to(device)
    return x, y


@torch.no_grad()
def estimate_split_loss(model, tokens, batch_size, block_size, n_batches, device):
    model.eval()
    losses = []
    for _ in range(n_batches):
        x, y = sample_batch(tokens, batch_size, block_size, device)
        logits, _ = model(x, return_attention=False)
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), y.view(B * T))
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


set_seed(CFG.seed)
model = TransformerLM(pilot_model_config).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.max_lr, betas=(0.9, 0.95), weight_decay=CFG.weight_decay)
scheduler = CosineScheduler(
    max_lr=CFG.max_lr,
    min_lr=CFG.min_lr,
    warmup_steps=CFG.warmup_steps,
    total_steps=CFG.total_steps,
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

history = []
train_loss_window = []
best_val_loss = float('inf')
best_step = None
no_improve_ckpts = 0
t0 = time.time()

initial_val_loss = estimate_split_loss(
    model,
    val_tokens,
    batch_size=CFG.batch_size,
    block_size=CFG.block_size,
    n_batches=CFG.train_eval_batches,
    device=DEVICE,
)
save_checkpoint(model, optimizer, 0, float('inf'), initial_val_loss, CFG.ckpt_dir, CFG.seed)
history.append({'step': 0, 'train_loss': float('inf'), 'val_loss': initial_val_loss})
best_val_loss = initial_val_loss
best_step = 0
ckpt0_path = CFG.ckpt_dir / 'ckpt_0000000.pt'
shutil.copy2(ckpt0_path, CFG.best_ckpt_path)
torch.save({'step': best_step, 'val_loss': best_val_loss}, CFG.best_ckpt_path.with_suffix('.meta.pt'))
print(f'step 0 | val_loss={initial_val_loss:.4f} | checkpoint saved [best]')

for step in range(1, CFG.total_steps + 1):
    lr = scheduler.set_lr(optimizer, step)
    x, y = sample_batch(train_tokens, CFG.batch_size, CFG.block_size, DEVICE)

    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
        logits, _ = model(x, return_attention=False)
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), y.view(B * T))

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
    scaler.step(optimizer)
    scaler.update()

    train_loss_window.append(loss.item())

    if step in CFG.checkpoint_steps:
        smooth_train_loss = float(np.mean(train_loss_window[-50:]))
        val_loss = estimate_split_loss(
            model,
            val_tokens,
            batch_size=CFG.batch_size,
            block_size=CFG.block_size,
            n_batches=CFG.train_eval_batches,
            device=DEVICE,
        )
        save_checkpoint(model, optimizer, step, smooth_train_loss, val_loss, CFG.ckpt_dir, CFG.seed)
        history.append({'step': step, 'train_loss': smooth_train_loss, 'val_loss': val_loss, 'lr': lr})
        ckpt_path = CFG.ckpt_dir / f'ckpt_{step:07d}.pt'
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_step = step
            no_improve_ckpts = 0
            shutil.copy2(ckpt_path, CFG.best_ckpt_path)
            torch.save({'step': best_step, 'val_loss': best_val_loss}, CFG.best_ckpt_path.with_suffix('.meta.pt'))
            best_marker = ' [best]'
        else:
            no_improve_ckpts += 1
            best_marker = ''
        elapsed = time.time() - t0
        print(f'step {step:5d} | train_loss={smooth_train_loss:.4f} | val_loss={val_loss:.4f} | best_val={best_val_loss:.4f} | lr={lr:.2e} | elapsed={elapsed/60:.1f} min{best_marker}')
        if step >= CFG.min_steps_before_early_stop and no_improve_ckpts >= CFG.early_stopping_patience_ckpts:
            print(f'Early stopping at step {step} after {no_improve_ckpts} non-improving checkpoints. Best checkpoint is step {best_step} with val_loss={best_val_loss:.4f}.')
            break

print('Training complete. Checkpoints saved to:', CFG.ckpt_dir)
print('Best validation checkpoint:', CFG.best_ckpt_path)
print(f'Best step: {best_step} | best val_loss: {best_val_loss:.4f}')


In [ ]:
history_steps = [row['step'] for row in history]
history_train = [row['train_loss'] if math.isfinite(row['train_loss']) else np.nan for row in history]
history_val = [row['val_loss'] for row in history]

plt.figure(figsize=(8, 4))
plt.plot(history_steps, history_train, marker='o', label='smoothed train loss')
plt.plot(history_steps, history_val, marker='o', label='validation loss')
if best_step is not None:
    plt.axvline(best_step, color='tab:green', linestyle='--', alpha=0.8, label=f'best step = {best_step}')
plt.title('Preliminary Training Curves')
plt.xlabel('checkpoint step')
plt.ylabel('loss')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Probe Every Saved Checkpoint

This cell reuses the repo's checkpoint format, attention extraction logic, scoring functions, and classifier.

Because this is a pilot:

- the model is smaller
- the checkpoint count is smaller
- the probe set is smaller

But the overall procedure is the same as the main experiment.


In [ ]:
ckpt_files = discover_checkpoints(CFG.ckpt_dir)
classifier = HeadClassifier(
    n_checkpoints=len(ckpt_files),
    n_layers=pilot_model_config.n_layers,
    n_heads=pilot_model_config.n_heads,
    seed=CFG.seed,
    ties_log_path=CFG.ties_path,
    thresholds=thresholds_mean,
)

for ckpt_idx, ckpt_path in enumerate(ckpt_files):
    extraction = extract_checkpoint(
        ckpt_path=ckpt_path,
        probe_dict=probe_dict,
        device=DEVICE,
        batch_size=CFG.probe_batch_size,
    )
    classifier.register_step(extraction.step)
    score_all_heads(
        extraction=extraction,
        probe_dict=probe_dict,
        classifier=classifier,
        ckpt_idx=ckpt_idx,
        step=extraction.step,
    )
    print(f'processed checkpoint {ckpt_idx + 1}/{len(ckpt_files)} at step {extraction.step}')

classifier.save(CFG.results_path)
print('Saved pilot results to:', CFG.results_path)
print('Saved tie log to:', CFG.ties_path)


## Analyze the Trajectories

This is a **single-seed production run**.

With `8 x 8 = 64` heads, the onset rule of `>= 5%` means a type needs about **3-4 heads** before it counts as having appeared.

What to look for:

- do sinks appear early?
- do previous-token heads appear at all?
- do non-UNDIFFERENTIATED heads increase over time?
- do any individual heads clearly change role over training?


In [ ]:
results = HeadClassifier.load(CFG.results_path)
global_curves = compute_global_curves([results])
onset_steps = compute_specialization_onset(global_curves, threshold_frac=0.05)
head_trajectories = compute_head_trajectories(results)
interesting = find_interesting_trajectories(head_trajectories, min_type_changes=2)

fig, ax = plt.subplots(figsize=(10, 5))
steps = global_curves['steps']
mean = global_curves['mean']

for type_idx, type_name in enumerate(global_curves['type_names']):
    if type_name == 'UNDIFFERENTIATED':
        continue
    ax.plot(steps, mean[:, type_idx], linewidth=2, label=type_name)

ax.set_title('Pilot Head-Type Fractions Across Training')
ax.set_xlabel('Training step')
ax.set_ylabel('Fraction of heads')
ax.grid(alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(CFG.figure_path, dpi=200)
plt.show()

print('Onset steps (>= 5% of heads):')
for name, step in onset_steps.items():
    print(f'  {name:18s}: {step}')

print('\nInteresting head trajectories (at least 2 type changes):', len(interesting))
for idx, ((layer, head), traj) in enumerate(list(interesting.items())[:8], start=1):
    readable = [HEAD_TYPES[t] for t in traj]
    print(f'  {idx}. layer={layer}, head={head} -> {readable}')

print('\nFigure saved to:', CFG.figure_path)


## How to Read This Production Run

Use this notebook as a **validation check** before launching the full multi-seed OpenWebText experiment.

If the run shows:

- some early sink or previous-token structure,
- some increase in specialization over checkpoints,
- at least a few heads with stable or changing identities,

then the full-scale multi-seed run is justified.

## Natural Next Steps

To scale this up after this production notebook:

1. move from WikiText-103 to the full OpenWebText pipeline in the repo,
2. run multiple seeds,
3. compare the single-seed curves against the multi-seed results.
